# CineMatch — Movie Recommender System
### Exploratory Data Analysis, Feature Engineering & Cosine Similarity
**Author:** Sahil (Reg: 2427010355)

This notebook demonstrates the end-to-end machine learning pipeline:
1. Ingesting TMDB movie metadata and credits
2. Extracting cast, director, genres, and keywords
3. Synthesizing contextual text tags
4. Vectorizing narrative features (CountVectorizer & Cosine Similarity)
5. Generating top-N movie recommendations
6. Exporting serialized artifacts (`movie_list.pkl`, `similarity.pkl`)

## 1. Import Dependencies & Load Datasets

In [1]:
import os
import ast
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define file paths (local workspace with Kaggle fallback)
movies_path = 'data/tmdb_5000_movies.csv' if os.path.exists('data/tmdb_5000_movies.csv') else '/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv'
credits_path = 'data/tmdb_5000_credits.csv' if os.path.exists('data/tmdb_5000_credits.csv') else '/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv'

movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

print(f"Loaded {movies.shape[0]} movies and {credits.shape[0]} credits records.")
movies.head(2)

Loaded 4803 movies and 4803 credits records.


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


## 2. Dataset Merging & Column Selection

In [2]:
# Merge datasets on movie title
movies = movies.merge(credits, on='title')

# Select relevant content attributes
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)
print(f"Cleaned dataset: {len(movies)} motion pictures.")
movies.head(3)

Cleaned dataset: 4806 motion pictures.


/var/folders/ds/h79w0k7d3cx65yn6ztpjvbdr0000gn/T/ipykernel_33031/4034389698.py:6: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  movies.dropna(inplace=True)
/var/folders/ds/h79w0k7d3cx65yn6ztpjvbdr0000gn/T/ipykernel_33031/4034389698.py:7: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  movies.reset_index(drop=True, inplace=True)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


## 3. Feature Extraction & Parsing (JSON AST)

In [3]:
def convert(text):
    """Extract names from serialized JSON lists."""
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

def extract_top3_cast(text):
    """Extract the top 3 credited actors."""
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

def fetch_director(text):
    """Extract the lead director from crew records."""
    L = []
    for i in ast.literal_eval(text):
        if i.get('job') == 'Director':
            L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(extract_top3_cast)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies.head(3)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]


## 4. Entity Collapse & Tag Synthesis

In [4]:
def collapse(L):
    """Remove whitespace between compound names (e.g. 'Sam Worthington' -> 'SamWorthington')."""
    return [i.replace(' ', '') for i in L]

movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

# Combine all textual features into a unified narrative 'tags' representation
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

new_df = movies[['movie_id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

print(f"Synthesized catalog size: {len(new_df)} titles.")
new_df.head(3)

Synthesized catalog size: 4806 titles.


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


## 5. Text Vectorization & Cosine Similarity Matrix

In [5]:
# Bag-of-Words Vectorization
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

print(f"Feature Matrix shape: {vectors.shape}")

# Compute pairwise cosine similarity
similarity = cosine_similarity(vectors)
print(f"Similarity Gram Matrix shape: {similarity.shape}")

Feature Matrix shape: (4806, 5000)


Similarity Gram Matrix shape: (4806, 4806)


## 6. Recommendation Engine & Verification

In [6]:
def recommend(movie_title, top_n=5):
    """Retrieve top-N recommendations for a given movie title."""
    matches = new_df[new_df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        print(f"Movie '{movie_title}' not found in catalog!")
        return []
    
    idx = matches.index[0]
    distances = sorted(list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1])
    
    recs = []
    print(f"Top {top_n} Recommendations for '{matches.iloc[0]['title']}':")
    print("-" * 50)
    for rank, (i, score) in enumerate(distances[1 : top_n + 1], start=1):
        rec_title = new_df.iloc[i].title
        print(f"{rank}. {rec_title} (Similarity: {score*100:.1f}%)")
        recs.append(rec_title)
    print()
    return recs

# Test recommendations across diverse film archetypes
recommend('Gandhi')
recommend('Avatar')
recommend('The Dark Knight')

Top 5 Recommendations for 'Gandhi':
--------------------------------------------------
1. Gandhi, My Father (Similarity: 26.1%)
2. The Wind That Shakes the Barley (Similarity: 24.7%)
3. A Passage to India (Similarity: 22.8%)
4. Guiana 1838 (Similarity: 19.1%)
5. Ramanujan (Similarity: 17.5%)

Top 5 Recommendations for 'Avatar':
--------------------------------------------------
1. Titan A.E. (Similarity: 26.1%)
2. Small Soldiers (Similarity: 25.8%)
3. Ender's Game (Similarity: 25.1%)
4. Aliens vs Predator: Requiem (Similarity: 24.9%)
5. Independence Day (Similarity: 24.8%)

Top 5 Recommendations for 'The Dark Knight':
--------------------------------------------------
1. The Dark Knight Rises (Similarity: 42.4%)
2. Batman Begins (Similarity: 39.4%)
3. Batman Returns (Similarity: 31.9%)
4. Batman Forever (Similarity: 29.0%)
5. Batman & Robin (Similarity: 28.2%)



['The Dark Knight Rises',
 'Batman Begins',
 'Batman Returns',
 'Batman Forever',
 'Batman & Robin']

## 7. Model Serialization & Export for Streamlit App

In [7]:
import os
import pickle

os.makedirs('model', exist_ok=True)

# Export models for Streamlit web app
pickle.dump(new_df, open('model/movie_list.pkl', 'wb'))
pickle.dump(similarity, open('model/similarity.pkl', 'wb'))

# Also keep root copies for Kaggle compatibility
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print(f"Successfully serialized {len(new_df)} movies to 'model/movie_list.pkl' and 'model/similarity.pkl'!")

Successfully serialized 4806 movies to 'model/movie_list.pkl' and 'model/similarity.pkl'!
